# Provider Directory Data Validation & Quality Analysis

### Overview
This project focuses on provider directory data because accurate provider records directly affect claims processing, referrals, credentialing, compliance, and overall network operations in healthcare organizations. The dataset was intentionally synthesized to emulate real-world provider operations by combining common issues such as missing values, duplicate records, invalid NPIs, inconsistent credentialing dates, specialty naming conflicts, and provider status discrepancies that frequently appear in healthcare systems. 

The goal of the project is to simulate the type of validation and data quality work performed by provider operations and data teams before reporting, audits, or network decisions are made. Improving provider data quality reduces operational delays, strengthens reporting accuracy, and supports more reliable business workflows by ensuring that downstream teams are working from complete, standardized, and trustworthy records.

### Key Takeaways

* Exact duplicate rows were removed, while repeated names such as `Maria Lopez` were confirmed to be separate providers with different `Provider_ID` and `NPI_Number` values. This shows why provider matching should rely on true identifiers rather than provider names alone.
* Fields such as `Specialty`, `License_State`, and `Insurance_Network_Status` contained formatting issues, spelling errors, and mixed naming conventions. Standardizing these fields improves grouped analysis and reporting accuracy.
* ZIP code validation standardized valid records into 5-digit text format, but 434 records still contain missing ZIP values. These should remain flagged for review rather than being filled with assumptions.
* `Last_Credentialing_Review_Date` contained mixed date formats that affected time-based reporting. Most valid dates were standardized, whil e 370 invalid values remain flagged for manual review.
* Final NPI validation showed that 526 records still do not meet the required 10-digit structure. Since NPI numbers are critical for credentialing and payer matching, these remain a high-priority issue.

## Data Loading and Preliminary Inspection

In [6]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
# load dataset
df = pd.read_csv('data/provider_validation_dataset.csv')

### Preliminary Data Inspection

The analysis begins by focusing on fields most prone to reporting errors and those most directly tied to the business workflow. These include `Specialty`, `Credentialing_Status`, `Insurance_Network_Status`, `License_State`, `ZIP_Code`, `Provider_ID`, and `NPI_Number`, as errors in these fields can affect claims processing, reporting accuracy, provider matching, and provider network compliance.

Because several of these fields serve as operational identifiers or validation checkpoints, duplicate values, formatting inconsistencies, and missing records can create significant downstream risks across reporting and provider management workflows.

This initial assessment is used to understand how much data is available, which fields are numeric, text, or mixed, where missing values appear, and whether obvious inconsistencies are already visible before deeper validation begins.

In [22]:
# Shape and first 10 rows of the dataset
print(f"✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

df.head(10)

✅ Dataset loaded: 10,004 rows × 14 columns


,Provider_ID,NPI_Number,Provider_Name,Specialty,License_State,City,ZIP_Code,Credentialing_Status,Insurance_Network_Status,Claims_Denied_Last_90_Days,Avg_Claim_Processing_Days,Monthly_Referral_Volume,Last_Credentialing_Review_Date,Notes
0,PRV-1001,1000000001,Emily Davis,Orthopedics,New York,Dallas,90001.0,Active,Pending,0,13.9,135,04/25/2025,Credential file complete
1,PRV-1002,1000000002,Robert Jones,Dermatology,California,Buffalo,90001.0,Under Review,In Network,0,3.5,129,2025-09-16,Credential file complete
2,PRV-1003,1000000003,David Chen,Dermatology,TX,Houston,75201.0,Pending,Out of Network,8,15.0,13,2025-03-23,Address needs verification
3,PRV-1004,1000000004,John Smith,Primary Care,New York,San Diego,90001.0,Active,Out of Network,12,3.5,443,2025-06-26,Pending license renewal
4,PRV-1005,1000000005,Emily Davis,Primary Care,CA,Los Angeles,75201.0,Pending,Out of Network,9,15.3,326,2025-07-05,Missing insurance form
5,PRV-1006,1000000006,David Chen,Primary Care,TX,Houston,77001.0,Expired,Pending,3,8.1,242,22-11-2025,Missing insurance form
6,PRV-1007,1000000007,Maria Lopez,Orthopedics,Texas,San Diego,14201.0,Expired,Pending,2,11.7,97,2025-10-01,Missing insurance form
7,PRV-1008,1000000008,David Chen,Orthopedics,Texas,New York,75201.0,Active,In Network,7,13.0,441,01/29/2025,Credential file complete
8,PRV-1009,1000000009,Maria Lopez,Neurology,TX,Houston,14201.0,Pending,Pending,10,5.4,265,2025-07-22,Address needs verification
9,PRV-1010,1000000010,Maria Lopez,Neurology,California,Dallas,75201.0,Under Review,In Network,8,14.0,229,26-10-2025,Pending license renewal


In [8]:
df.dtypes

Provider_ID                        object
NPI_Number                          int64
Provider_Name                      object
Specialty                          object
License_State                      object
City                               object
ZIP_Code                          float64
Credentialing_Status               object
Insurance_Network_Status           object
Claims_Denied_Last_90_Days          int64
Avg_Claim_Processing_Days         float64
Monthly_Referral_Volume             int64
Last_Credentialing_Review_Date     object
Notes                              object
dtype: object

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10004 entries, 0 to 10003
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Provider_ID                     10004 non-null  object 
 1   NPI_Number                      10004 non-null  int64  
 2   Provider_Name                   10004 non-null  object 
 3   Specialty                       10004 non-null  object 
 4   License_State                   10004 non-null  object 
 5   City                            10004 non-null  object 
 6   ZIP_Code                        9570 non-null   float64
 7   Credentialing_Status            10004 non-null  object 
 8   Insurance_Network_Status        10004 non-null  object 
 9   Claims_Denied_Last_90_Days      10004 non-null  int64  
 10  Avg_Claim_Processing_Days       10004 non-null  float64
 11  Monthly_Referral_Volume         10004 non-null  int64  
 12  Last_Credentialing_Review_Date  

In [30]:
# Percentage of missing values for each column
missing_percent = (df.isnull().sum() / len(df)) * 100

print(missing_percent)

Provider_ID                       0.000000
NPI_Number                        0.000000
Provider_Name                     0.000000
Specialty                         0.000000
License_State                     0.000000
City                              0.000000
ZIP_Code                          4.338265
Credentialing_Status              0.000000
Insurance_Network_Status          0.000000
Claims_Denied_Last_90_Days        0.000000
Avg_Claim_Processing_Days         0.000000
Monthly_Referral_Volume           0.000000
Last_Credentialing_Review_Date    0.000000
Notes                             0.000000
dtype: float64


In [31]:
df.describe()

,NPI_Number,ZIP_Code,Claims_Denied_Last_90_Days,Avg_Claim_Processing_Days,Monthly_Referral_Volume
count,1.000400e+04,9570.000000,10004.000000,10004.000000,10004.000000
mean,9.479518e+08,59375.305120,12.139844,12.393673,309.406038
std,2.209706e+08,34470.565596,7.877710,15.282695,373.858129
min,1.000002e+07,10001.000000,-3.000000,2.000000,10.000000
25%,1.000002e+09,14201.000000,5.000000,6.000000,133.000000
50%,1.000005e+09,75201.000000,12.000000,10.100000,260.000000
75%,1.000007e+09,90001.000000,19.000000,14.300000,387.000000
max,1.000010e+09,92101.000000,25.000000,99.900000,2500.000000


#### Insights

* The `Last_Credentialing_Review_Date` column is stored in mixed string formats rather than a standardized datetime format, and some values appear invalid. This requires parsing and validation before any reliable time based analysis can be performed.
* `ZIP_Code` is currently stored as a float and contains missing values (~4.33% of total records). ZIP codes should be stored as text to preserve formatting consistency and prevent issues such as decimal formatting or the loss of leading zeros during validation.
* `df.describe()` highlights several immediate data quality concerns:
     * `Claims_Denied_Last_90_Days` contains a negative minimum value, which is likely invalid and suggests input or system entry errors.
     * `Monthly_Referral_Volume` shows a large gap between typical referral volume and the maximum recorded value, indicating likely outliers. In this case, the median may provide a more reliable summary than the mean when evaluating provider activity.
* Some fields may contain placeholder missing values stored as text rather than true null values (such as NULL, N/A, Unknown, or blank spaces). These values are not captured by standard `isna()` checks and can understate the true amount of missing data, requiring additional review before cleaning decisions are made.

### Categorical Validation

For the sake of brevity, this project focuses on the most impactful fields, particularly those with inconsistencies and errors that can quickly lead to unreliable analysis.

In [16]:
df["Specialty"].value_counts(dropna=False)

Specialty
Neurology        1509
Primary Care     1504
Pediatrics       1487
Dermatology      1480
Orthopedics      1473
Cardiology       1464
Cardiolgy         344
dermatology       147
neurology         137
primary care      119
cardiology        118
orthopedics       111
pediatrics        111
Name: count, dtype: int64

In [17]:
df["License_State"].value_counts(dropna=False)

License_State
New York      1719
CA            1703
NY            1657
TX            1657
California    1643
Texas         1625
Name: count, dtype: int64

In [29]:
df["Credentialing_Status"].value_counts(dropna=False)

Credentialing_Status
Under Review    2551
Active          2497
Pending         2480
Expired         2476
Name: count, dtype: int64

#### Insights
 
These fields exhibit multiple inconsistencies, including different versions of the same value caused by capitalization, spacing, and spelling issues under `Specialty`, along with formatting inconsistencies under `License_State`. Notably, `Credentialing_Status` appears to be consistent, which is important because this field helps determine whether a provider is cleared to safely and compliantly participate in the healthcare network and directly affects claims processing, insurance reimbursement, referrals, and compliance requirements.

Label inconsistencies such as `Cardiology`, `cardiology`, and `Cardiolgy` are currently treated as separate values, causing grouped analysis by `Specialty` to become inaccurate and unreliable when the field should contain a limited set of clean, standardized values.

Further `value_counts()` review also surfaced two structural issues outside the categorical fields:    
* `ZIP_Code` is currently stored as a float and should be converted to a standardized text format for consistency and accurate validation.
* `Provider_ID` and `NPI_Number` show signs of duplicate records. The Length: 10000 output suggests approximately 4 duplicate provider entries across the 10,004 total rows, which will be reviewed further to determine whether these represent exact duplicate rows or conflicting records sharing the same identifier.

### Identifier Review

`Provider_ID` and `NPI_Number` serve as the primary identifiers used to track providers across credentialing, claims processing, reporting, and compliance workflows. Because these fields function as operational anchors across multiple systems, duplicate values, missing identifiers, or malformed NPIs can create significant downstream risks.

This section focuses on validating identifier completeness, uniqueness, and structural integrity to determine whether provider records can be reliably matched and trusted across business operations.

#### Provider_ID Validation

In [46]:
# Check Missing identifiers
df["Provider_ID"].isnull().sum()

np.int64(0)

A provider record without an ID can cause workflow problems with claim routing, credentialing validation, reimbursement tracking and general provider matching across systems. This is checked first because missing identifiers create immediate breakdowns in provider tracking across operational systems.

In [47]:
# Check duplicate identifiers
print("Duplicate Provider_IDs:", df["Provider_ID"].duplicated().sum())

Duplicate Provider_IDs: 4


In [35]:
# Inspect `Provider_ID` duplicates
df[df["Provider_ID"].duplicated(keep=False)].sort_values("Provider_ID")

,Provider_ID,NPI_Number,Provider_Name,Specialty,License_State,City,ZIP_Code,Credentialing_Status,Insurance_Network_Status,Claims_Denied_Last_90_Days,Avg_Claim_Processing_Days,Monthly_Referral_Volume,Last_Credentialing_Review_Date,Notes
100,PRV-1101,1000000101,John Smith,Neurology,Texas,Los Angeles,10001.0,Pending,In Network,5,6.1,432,04-09-2025,Pending license renewal
10000,PRV-1101,1000000101,John Smith,Neurology,Texas,Los Angeles,10001.0,Pending,In Network,5,6.1,432,04-09-2025,Pending license renewal
500,PRV-1501,1000000501,Maria Lopez,Primary Care,TX,New York,92101.0,Active,Pending,11,4.5,145,2025-07-07,Missing insurance form
10001,PRV-1501,1000000501,Maria Lopez,Primary Care,TX,New York,92101.0,Active,Pending,11,4.5,145,2025-07-07,Missing insurance form
2500,PRV-3501,1000002501,Maria Lopez,Pediatrics,NY,Dallas,77001.0,Pending,In Network,0,2.6,2500,06-05-2025,Pending license renewal
10002,PRV-3501,1000002501,Maria Lopez,Pediatrics,NY,Dallas,77001.0,Pending,In Network,0,2.6,2500,06-05-2025,Pending license renewal
7777,PRV-8778,1000007778,Maria Lopez,Neurology,Texas,Dallas,77001.0,Pending,Pending,0,15.4,476,28-03-2025,Contract under review
10003,PRV-8778,1000007778,Maria Lopez,Neurology,Texas,Dallas,77001.0,Pending,Pending,0,15.4,476,28-03-2025,Contract under review


#### NPI_Number Validation

**1.) Length Validation**

In [55]:
# Check missing identifiers
df["NPI_Number"].isnull().sum()

np.int64(0)

In [50]:
# Check duplicate identifiers
print("Duplicate NPI_Numbers:", df["NPI_Number"].duplicated().sum())

Duplicate NPI_Numbers: 4


In [37]:
# Inspect `NPI_Number` duplicates
df[df["NPI_Number"].duplicated(keep=False)].sort_values("NPI_Number")

,Provider_ID,NPI_Number,Provider_Name,Specialty,License_State,City,ZIP_Code,Credentialing_Status,Insurance_Network_Status,Claims_Denied_Last_90_Days,Avg_Claim_Processing_Days,Monthly_Referral_Volume,Last_Credentialing_Review_Date,Notes
100,PRV-1101,1000000101,John Smith,Neurology,Texas,Los Angeles,10001.0,Pending,In Network,5,6.1,432,04-09-2025,Pending license renewal
10000,PRV-1101,1000000101,John Smith,Neurology,Texas,Los Angeles,10001.0,Pending,In Network,5,6.1,432,04-09-2025,Pending license renewal
10001,PRV-1501,1000000501,Maria Lopez,Primary Care,TX,New York,92101.0,Active,Pending,11,4.5,145,2025-07-07,Missing insurance form
500,PRV-1501,1000000501,Maria Lopez,Primary Care,TX,New York,92101.0,Active,Pending,11,4.5,145,2025-07-07,Missing insurance form
10002,PRV-3501,1000002501,Maria Lopez,Pediatrics,NY,Dallas,77001.0,Pending,In Network,0,2.6,2500,06-05-2025,Pending license renewal
2500,PRV-3501,1000002501,Maria Lopez,Pediatrics,NY,Dallas,77001.0,Pending,In Network,0,2.6,2500,06-05-2025,Pending license renewal
7777,PRV-8778,1000007778,Maria Lopez,Neurology,Texas,Dallas,77001.0,Pending,Pending,0,15.4,476,28-03-2025,Contract under review
10003,PRV-8778,1000007778,Maria Lopez,Neurology,Texas,Dallas,77001.0,Pending,Pending,0,15.4,476,28-03-2025,Contract under review


In [63]:
# NPI length validation (10-digit standard validation)
df["NPI_Length"] = (
    df["NPI_Number"]
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.len()
)

df["NPI_Length"].value_counts(dropna=False)

NPI_Length
10    9478
8      526
Name: count, dtype: int64

In [51]:
# Isolate inconsistencies across selected columns
df[df["NPI_Length"] != 10][["Provider_ID", "NPI_Number", "NPI_Length"]]

,Provider_ID,NPI_Number,NPI_Length
18,PRV-1019,10000019,8
37,PRV-1038,10000038,8
56,PRV-1057,10000057,8
75,PRV-1076,10000076,8
94,PRV-1095,10000095,8
...,...,...,...
9917,PRV-10918,10009918,8
9936,PRV-10937,10009937,8
9955,PRV-10956,10009956,8
9974,PRV-10975,10009975,8


**2.) Numeric Content Validation**

In [60]:
# Non-Numeric NPI check
df["NPI_Number"].astype(str).str.isdigit().value_counts()

NPI_Number
True    10004
Name: count, dtype: int64

In [61]:
# Check numerical accuracy of NPI values
df[
    ~df["NPI_Number"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.isdigit()
][["Provider_ID", "NPI_Number"]]

,Provider_ID,NPI_Number


In [62]:
# Exact duplicate vs conflicting record check
duplicate_providers = df[df["Provider_ID"].duplicated(keep=False)]

duplicate_providers.sort_values("Provider_ID")

,Provider_ID,NPI_Number,Provider_Name,Specialty,License_State,City,ZIP_Code,Credentialing_Status,Insurance_Network_Status,Claims_Denied_Last_90_Days,Avg_Claim_Processing_Days,Monthly_Referral_Volume,Last_Credentialing_Review_Date,Notes,NPI_Length
100,PRV-1101,1000000101,John Smith,Neurology,Texas,Los Angeles,10001.0,Pending,In Network,5,6.1,432,04-09-2025,Pending license renewal,10
10000,PRV-1101,1000000101,John Smith,Neurology,Texas,Los Angeles,10001.0,Pending,In Network,5,6.1,432,04-09-2025,Pending license renewal,10
500,PRV-1501,1000000501,Maria Lopez,Primary Care,TX,New York,92101.0,Active,Pending,11,4.5,145,2025-07-07,Missing insurance form,10
10001,PRV-1501,1000000501,Maria Lopez,Primary Care,TX,New York,92101.0,Active,Pending,11,4.5,145,2025-07-07,Missing insurance form,10
2500,PRV-3501,1000002501,Maria Lopez,Pediatrics,NY,Dallas,77001.0,Pending,In Network,0,2.6,2500,06-05-2025,Pending license renewal,10
10002,PRV-3501,1000002501,Maria Lopez,Pediatrics,NY,Dallas,77001.0,Pending,In Network,0,2.6,2500,06-05-2025,Pending license renewal,10
7777,PRV-8778,1000007778,Maria Lopez,Neurology,Texas,Dallas,77001.0,Pending,Pending,0,15.4,476,28-03-2025,Contract under review,10
10003,PRV-8778,1000007778,Maria Lopez,Neurology,Texas,Dallas,77001.0,Pending,Pending,0,15.4,476,28-03-2025,Contract under review,10


#### Insights

* `Provider_ID` and `NPI_Number` both show duplicate values, confirming that some provider records are not uniquely identifiable. This creates risk for claims processing, reporting accuracy, and provider matching because multiple records may be treated as the same provider or the same provider may be counted more than once.
* Review of duplicate `Provider_ID` records shows that some duplicates are not exact row copies. Providers such as `Maria Lopez` contain conflicting surrounding information across records, suggesting the issue goes beyond simple duplicate removal and may reflect record conflicts, system merge issues, or incorrect identifier assignment.
* `NPI_Number` length validation shows that not all records follow the required 10 digit structure. Since NPIs function as standardized external provider identifiers, incorrect length can prevent accurate credentialing checks, matching, and reimbursement workflows.
* Non-numeric validation returned no clearly malformed NPI values after removing float formatting artifacts such as `.0`, signaling that identifier issues are more likely tied to duplicate records or incorrect length rather than letters, symbols, or obvious formatting errors.

## Data Cleaning and Validation

After identifying issues in the preliminary review, the next step is to correct the inconsistencies and prepare the dataset for reliable reporting and operational use.

This section focuses on improving data consistency, resolving duplicate records, validating identifiers, and correcting structural issues that affect provider matching, claims workflows, and reporting accuracy.

### Categorical Standardization

In [69]:
# Standardize `Specialty`
df["Specialty_Clean"] = (
    df["Specialty"]
    .str.strip()            # Remove extra spacing
    .str.title()            # Fix capitalization
)

# Standardize License_State
df["License_State_Clean"] = (
    df["License_State"]
    .str.strip()           
    .str.upper()           # Standardize state abbreviations
)

# Standardize `Insurance_Network_Status`
df["Insurance_Network_Status_Clean"] = (
    df["Insurance_Network_Status"]
    .str.strip()
    .str.title()
)

In [70]:
# Correct known specialty issues
df["Specialty_Clean"] = df["Specialty_Clean"].replace({
    "Cardiolgy": "Cardiology"
})

In [71]:
# Convert full state names to abbreviations
df["License_State_Clean"] = df["License_State_Clean"].replace({
    "CALIFORNIA": "CA",
    "NEW YORK": "NY",
    "TEXAS": "TX"
})

**Review**

In [72]:
df["Specialty_Clean"].value_counts(dropna=False)

Specialty_Clean
Cardiology      1926
Neurology       1646
Dermatology     1627
Primary Care    1623
Pediatrics      1598
Orthopedics     1584
Name: count, dtype: int64

In [73]:
df["License_State_Clean"].value_counts(dropna=False)

License_State_Clean
NY    3376
CA    3346
TX    3282
Name: count, dtype: int64

In [74]:
df["Insurance_Network_Status_Clean"].value_counts(dropna=False)

Insurance_Network_Status_Clean
In Network        3388
Out Of Network    3358
Pending           3258
Name: count, dtype: int64

#### Insights

Basic formatting cleanup removed spacing and capitalization issues, but manual standardization is required for true label inconsistencies such as `Cardiolgy` and full state names like `CALIFORNIA` and `NEW YORK`. This ensures grouped analysis reflects real categories instead of multiple versions of the same value.

### Placeholder Missing Value Cleanup

Some missing values, like `NULL`, `N/A`, `Unkown`, and blank spaces are stored as text instead of true null values and are not detected by standard `isna()` checks.

In [76]:
# Common placeholder missing values
placeholder_values = [
    "NULL",
    "N/A",
    "Unknown",
    "unknown",
    "",
    " "
]

# Replace with true null values
df = df.replace(
    placeholder_values,
    np.nan
)

In [77]:
missing_percent_after = (
    df.isnull().sum() / len(df)
) * 100

print(missing_percent_after)

Provider_ID                       0.000000
NPI_Number                        0.000000
Provider_Name                     0.000000
Specialty                         0.000000
License_State                     0.000000
City                              0.000000
ZIP_Code                          4.338265
Credentialing_Status              0.000000
Insurance_Network_Status          0.000000
Claims_Denied_Last_90_Days        0.000000
Avg_Claim_Processing_Days         0.000000
Monthly_Referral_Volume           0.000000
Last_Credentialing_Review_Date    0.000000
Notes                             0.000000
NPI_Length                        0.000000
Specialty_Clean                   0.000000
License_State_Clean               0.000000
Insurance_Network_Status_Clean    0.000000
dtype: float64


#### Insights

Rechecking missing values after placeholder cleanup showed little change from the initial assessment, suggesting that most missing data was already stored as true null values rather than text placeholders such as `NULL` or `Unknown`. `ZIP_Code` remains the primary field with meaningful missingness at approximately 4.33%, while the rest of the dataset shows strong completeness across key operational fields.

### ZIP Code Formatting

ZIP codes should be stored as text identifiers, not floats for calculation otherwise numeric values in this field can cause decimal formatting issues, loss of leading zeros, and inconsistent validation.

In [78]:
# Convert `ZIP_Code` to 5-digit text
df["ZIP_Code_Clean"] = df["ZIP_Code"].apply(
    lambda x: str(int(x)).zfill(5)           # Convert to text, remove `.0`, and 5-digit standardize
    if pd.notna(x)
    else np.nan
)

In [81]:
# Check code length excluding missing values
df[
    ["ZIP_Code", "ZIP_Code_Clean"]
].head(10)

df["ZIP_Code_Clean"].dropna().str.len().value_counts

<bound method IndexOpsMixin.value_counts of 0        5
1        5
2        5
3        5
4        5
        ..
9999     5
10000    5
10001    5
10002    5
10003    5
Name: ZIP_Code_Clean, Length: 9570, dtype: int64>

#### Insight

ZIP code formatting is standardized for valid records, but 434 provider records still contain missing ZIP values. Since ZIP codes affect provider location accuracy, referrals, reporting, and network operations, these records should stay flagged for manual review instead of being filled in with assumptions. If reliable provider history or matching location records are available, some values may be recovered. Otherwise, the missing values should remain in place until they can be properly verified.

### Date Validation

In order to be used for time-based reporting, `Last_Credentialing_Review_Date` needs to be stored as a true datetime field.

Mixed formats or invalid dates cause inaccurate reporting and make it difficult to review timelines correctly.

In [88]:
# Convert to datetime format
df["Last_Credentialing_Review_Date_Clean"] = pd.to_datetime(   # Convert to datetime
    df["Last_Credentialing_Review_Date"],         
    errors="coerce"                               # turn invalid dates into NaT (not a valid time)
)

In [87]:
df["Last_Credentialing_Review_Date_Clean"].isna().sum()

np.int64(6843)

This returned a high number of failed conversions, so the next step would be to inspect what actually failed and why.

In [97]:
invalid_date_records = df[
    df["Last_Credentialing_Review_Date_Clean"].isna()
]

invalid_date_records[
    "Last_Credentialing_Review_Date"
].value_counts(dropna=False).head(30)

Last_Credentialing_Review_Date
13/32/2025    370
14-11-2025     18
27-01-2025     18
23-09-2025     17
2025-12-05     17
24-11-2025     17
2025-06-27     17
14-03-2025     17
2025-07-30     16
17-12-2025     16
2025-11-03     16
2025-06-02     16
2025-12-18     16
25-02-2025     16
07-06-2025     15
2025-12-04     15
2025-11-07     15
08-08-2025     15
25-04-2025     15
2025-09-17     15
20-10-2025     15
2025-08-11     15
2025-11-13     14
20-03-2025     14
2025-09-07     14
30-12-2025     14
2025-02-11     14
14-05-2025     14
2025-10-29     14
15-02-2025     14
Name: count, dtype: int64

In [109]:
# Convert mixed-format dates
df["Last_Credentialing_Review_Date_Clean"] = pd.to_datetime(
    df["Last_Credentialing_Review_Date"]
        .astype(str)
        .str.strip(),
    errors="coerce",
    format="mixed"
)

In [107]:
# Recheck
df["Last_Credentialing_Review_Date_Clean"].isna().sum()

np.int64(370)

These are invalid source values, such as impossible dates. They should not be guessed or manually “fixed” without a reliable source and will be flagged for manual review or source-system correction.

In [110]:
# Inspect remaining invalid dates
remaining_invalid_dates = df[
    df["Last_Credentialing_Review_Date_Clean"].isna()
]

remaining_invalid_dates[
    [
        "Provider_ID",
        "Provider_Name",
        "Last_Credentialing_Review_Date"
    ]
].head(20)

,Provider_ID,Provider_Name,Last_Credentialing_Review_Date
26,PRV-1027,Emily Davis,13/32/2025
53,PRV-1054,Robert Jones,13/32/2025
80,PRV-1081,David Chen,13/32/2025
107,PRV-1108,Emily Davis,13/32/2025
134,PRV-1135,Sarah Kim,13/32/2025
161,PRV-1162,David Chen,13/32/2025
188,PRV-1189,John Smith,13/32/2025
215,PRV-1216,Maria Lopez,13/32/2025
242,PRV-1243,Sarah Kim,13/32/2025
269,PRV-1270,Robert Jones,13/32/2025


#### Insight

Date validation showed that `Last_Credentialing_Review_Date` contained mixed date formats rather than one consistent structure. Using `format="mixed"` allowed valid date values to be converted into a standardized datetime format, while values that still could not be converted remained flagged for review. This makes the field more reliable for credentialing timelines, compliance tracking, and time-based reporting without forcing corrections on invalid source values.

### Invalid Numeric Value Review

During preliminary analysis, `df.describe()` highlighted several values that do not make practical business sense, especially negative counts and unusually high values. This section focuses on reviewing those atypical records to determine whether they reflect input errors, true outliers, or values that require further validation before reporting.

**Negative Denied Claims Review**

In [111]:
# Check negative denied claims
negative_claims = df[
    df["Claims_Denied_Last_90_Days"] < 0
]

negative_claims[
    [
        "Provider_ID",
        "Provider_Name",
        "Claims_Denied_Last_90_Days"
    ]
]

,Provider_ID,Provider_Name,Claims_Denied_Last_90_Days
30,PRV-1031,Maria Lopez,-3
61,PRV-1062,Robert Jones,-3
92,PRV-1093,David Chen,-3
123,PRV-1124,Maria Lopez,-3
154,PRV-1155,Sarah Kim,-3
...,...,...,...
9857,PRV-10858,Robert Jones,-3
9888,PRV-10889,John Smith,-3
9919,PRV-10920,Emily Davis,-3
9950,PRV-10951,Sarah Kim,-3


In [117]:
# Count
print("Total number of affected records:", negative_claims.shape[0])

Total number of affected records: 322


In [116]:
# Review referral volume distribution
df["Monthly_Referral_Volume"].describe()

count    10004.000000
mean       309.406038
std        373.858129
min         10.000000
25%        133.000000
50%        260.000000
75%        387.000000
max       2500.000000
Name: Monthly_Referral_Volume, dtype: float64

In [118]:
# Review highest referral volume records
df.sort_values(
    "Monthly_Referral_Volume",
    ascending=False
)[
    [
        "Provider_ID",
        "Provider_Name",
        "Specialty",
        "Monthly_Referral_Volume"
    ]
].head(20)

,Provider_ID,Provider_Name,Specialty,Monthly_Referral_Volume
1557,PRV-2558,David Chen,Pediatrics,2500
10002,PRV-3501,Maria Lopez,Pediatrics,2500
6395,PRV-7396,Maria Lopez,neurology,2500
6354,PRV-7355,John Smith,Dermatology,2500
655,PRV-1656,Emily Davis,Cardiology,2500
6313,PRV-7314,David Chen,Neurology,2500
1598,PRV-2599,Sarah Kim,orthopedics,2500
6518,PRV-7519,Sarah Kim,Neurology,2500
9921,PRV-10922,Sarah Kim,Neurology,2500
9962,PRV-10963,Robert Jones,Cardiology,2500


In [119]:
# Compare mean and median referral volume
mean_referrals = df["Monthly_Referral_Volume"].mean()
median_referrals = df["Monthly_Referral_Volume"].median()

print("Mean:", mean_referrals)
print("Median:", median_referrals)

Mean: 309.406037584966
Median: 260.0


#### Insight

Numeric validation identified values that require review before reporting. Negative denied claim counts are likely input errors because claim counts should not fall below zero. Referral volume also shows unusually high values, which may represent high-activity providers or possible outliers. Since extreme values can pull the average upward, the median may provide a more reliable summary of typical provider activity.

### Duplicate Record Resolution

This step addresses the duplicate records identified during preliminary data inspection by separating simple duplicate rows from more serious conflicting provider records.

Earlier review showed 4 exact duplicate rows and repeated provider names such as `Maria Lopez`, which initially suggests possible provider identity conflicts. The goal of this section is to determine whether these duplicates reflect exact row repetition or true identifier conflicts across `Provider_ID`.

In [123]:
# Remove exact duplicate rows
df = df.drop_duplicates()

In [124]:
# Review
duplicate_provider_records = df[
    df["Provider_ID"].duplicated(keep=False)
]

duplicate_provider_records.sort_values("Provider_ID")

,Provider_ID,NPI_Number,Provider_Name,Specialty,License_State,City,ZIP_Code,Credentialing_Status,Insurance_Network_Status,Claims_Denied_Last_90_Days,...,Monthly_Referral_Volume,Last_Credentialing_Review_Date,Notes,NPI_Length,Specialty_Clean,License_State_Clean,Insurance_Network_Status_Clean,ZIP_Code_Clean,Last_Credentialing_Date,Last_Credentialing_Review_Date_Clean


In [126]:
# Compare accross duplicate providers
duplicate_provider_records.groupby("Provider_ID").nunique()

,NPI_Number,Provider_Name,Specialty,License_State,City,ZIP_Code,Credentialing_Status,Insurance_Network_Status,Claims_Denied_Last_90_Days,Avg_Claim_Processing_Days,Monthly_Referral_Volume,Last_Credentialing_Review_Date,Notes,NPI_Length,Specialty_Clean,License_State_Clean,Insurance_Network_Status_Clean,ZIP_Code_Clean,Last_Credentialing_Date,Last_Credentialing_Review_Date_Clean
Provider_ID,,,,,,,,,,,,,,,,,,,,


In [130]:
# Compare key fields for duplicated records
duplicate_provider_records.groupby("Provider_ID")[
    [
        "Provider_Name",
        "NPI_Number",
        "Specialty",
        "License_State",
        "Credentialing_Status",
        "Insurance_Network_Status"
    ]
].nunique()

,Provider_Name,NPI_Number,Specialty,License_State,Credentialing_Status,Insurance_Network_Status
Provider_ID,,,,,,


In [131]:
duplicate_provider_records[
    duplicate_provider_records["Provider_Name"] == "Maria Lopez"
].sort_values("Provider_ID")

,Provider_ID,NPI_Number,Provider_Name,Specialty,License_State,City,ZIP_Code,Credentialing_Status,Insurance_Network_Status,Claims_Denied_Last_90_Days,...,Monthly_Referral_Volume,Last_Credentialing_Review_Date,Notes,NPI_Length,Specialty_Clean,License_State_Clean,Insurance_Network_Status_Clean,ZIP_Code_Clean,Last_Credentialing_Date,Last_Credentialing_Review_Date_Clean


#### Insight

The duplicate issue was resolved by removing 4 exact duplicate rows with `drop_duplicates()`. Follow up checks confirmed that no duplicate `Provider_ID` records remained after removal, meaning the duplicate issue was limited to repeated row copies rather than conflicting provider identifiers. Repeated names such as `Maria Lopez` were not removed because they represented different provider records with distinct identifiers, reinforcing why provider matching should rely on `Provider_ID` and `NPI_Number` rather than name alone.

### Final Validation Check

The final step in this analysis confirms what was cleaned, what improved, and what still needs review.

Realistically speaking, final validation does not mean the dataset is perfect, however, it does mean that the remaining issues are clearly identified.

In [135]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   Provider_ID                           10000 non-null  object        
 1   NPI_Number                            10000 non-null  int64         
 2   Provider_Name                         10000 non-null  object        
 3   Specialty                             10000 non-null  object        
 4   License_State                         10000 non-null  object        
 5   City                                  10000 non-null  object        
 6   ZIP_Code                              9566 non-null   float64       
 7   Credentialing_Status                  10000 non-null  object        
 8   Insurance_Network_Status              10000 non-null  object        
 9   Claims_Denied_Last_90_Days            10000 non-null  int64         
 10  Avg_

* Exact duplicate rows were removed
* Cleaned columns were successfully created
* data types are now appropriate for analysis

In [136]:
# Final missing value percentages
final_missing_percent = (
    df.isnull().sum() / len(df)
) * 100

print(final_missing_percent)

Provider_ID                             0.00
NPI_Number                              0.00
Provider_Name                           0.00
Specialty                               0.00
License_State                           0.00
City                                    0.00
ZIP_Code                                4.34
Credentialing_Status                    0.00
Insurance_Network_Status                0.00
Claims_Denied_Last_90_Days              0.00
Avg_Claim_Processing_Days               0.00
Monthly_Referral_Volume                 0.00
Last_Credentialing_Review_Date          0.00
Notes                                   0.00
NPI_Length                              0.00
Specialty_Clean                         0.00
License_State_Clean                     0.00
Insurance_Network_Status_Clean          0.00
ZIP_Code_Clean                          4.34
Last_Credentialing_Review_Date_Clean    3.70
dtype: float64


* `ZIP_Code_Clean` still has 434 missing values
* invalid credentialing dates remain flagged through failed datetime conversion

In [137]:
# Confirm cleaned `Specialty` values
df["Specialty_Clean"].value_counts(dropna=False)

Specialty_Clean
Cardiology      1926
Neurology       1644
Dermatology     1627
Primary Care    1622
Pediatrics      1597
Orthopedics     1584
Name: count, dtype: int64

In [138]:
# Confirm cleaned `License_State` values
df["License_State_Clean"].value_counts(dropna=False)

License_State_Clean
NY    3375
CA    3346
TX    3279
Name: count, dtype: int64

In [139]:
# Confirm cleaned `Insurance_Network_Status` values
df["Insurance_Network_Status_Clean"].value_counts(dropna=False)

Insurance_Network_Status_Clean
In Network        3386
Out Of Network    3358
Pending           3256
Name: count, dtype: int64

* Capitalization issues were corrected
* Spacing inconsistencies were removed
* Spelling fixes were applied
* State names were standardized

In [140]:
# Confirm valid ZIP codes are 5 digits
df["ZIP_Code_Clean"].dropna().str.len().value_counts()

ZIP_Code_Clean
5    9566
Name: count, dtype: int64

In [141]:
# Confirm remaining missing ZIP codes
df["ZIP_Code_Clean"].isna().sum()

np.int64(434)

* The first check confirms valid ZIP codes were standardized into proper 5-digit text format.
*  The second confirms how many records still have missing ZIP values.

In [142]:
# Remaining invalid credentialing dates
df["Last_Credentialing_Review_Date_Clean"].isna().sum()

np.int64(370)

In [143]:
# Percentage of remaining invalid dates
(
    df["Last_Credentialing_Review_Date_Clean"].isna().sum() / len(df)
) * 100

np.float64(3.6999999999999997)

* This confirms how many credentialing review dates still could not be converted after mixed-format parsing.
* 370 remain invalid, should not be corrected by assumption, and should stay flagged for source verification.

In [144]:
# Final duplicate identifier review
print(
    "Duplicate Provider_IDs:",
    df["Provider_ID"].duplicated().sum()
)

print(
    "Duplicate NPI_Numbers:",
    df["NPI_Number"].duplicated().sum()
)

Duplicate Provider_IDs: 0
Duplicate NPI_Numbers: 0


In [145]:
# Final NPI length review
df["NPI_Length"].value_counts(dropna=False)

NPI_Length
10    9474
8      526
Name: count, dtype: int64

* No unresolved duplicate `Provider_ID` records remain
* `NPI_Number` structure remains valid
* Provider records can be reliably match across systems
* The remaining 526 records exhibiting NPI-digit structuring inconsistencies should be reviewed against a reliable source before the dataset is used for final reporting or operational decisions.

#### Final Insight

Final validation confirms that major formatting inconsistencies, categorical issues, duplicate rows, and mixed-format credentialing dates were successfully addressed. Remaining unresolved records, including 434 missing ZIP codes and 370 invalid credentialing review dates, remain intentionally flagged for manual review rather than being filled with assumptions. With provider identifiers now reliable and core reporting fields standardized, the dataset is significantly more trustworthy for reporting, provider matching, and operational decision-making.

## Business Recommendations

* Add stronger validation for fields like `NPI_Number`, `ZIP_Code`, and `Last_Credentialing_Review_Date` so incomplete or invalid records are caught during data entry.
* Replace free text entry for fields such as `Specialty`, `License_State`, and `Insurance_Network_Status` with standardized dropdown selections to reduce inconsistent values.
* Check NPI numbers against the required 10-digit structure during provider onboarding and record updates to prevent identifier issues downstream.
* Run duplicate checks using `Provider_ID` and `NPI_Number` before new provider records are created to prevent duplicate records from affecting reporting and claims workflows.
* Keep missing ZIP codes and invalid credentialing dates flagged until they can be verified from reliable provider records instead of filling them with assumptions.
* Review provider directory data regularly so issues are caught early and reporting remains reliable.